# Jamaica Pilot of 6-step decision-making methodology

#### This code is structured chronologically according to the 6 steps:

Step 1: Baseline assessment of natural assets

Step 2: Identifying existing infrastructure-related ecosystem services

Step 3: Assessment of threats to natural assets and their services

Step 4: Identifying nature-based solutions options for infrastructure

Step 5: Identification of costs and benefits

Step 6: Prioritization

## Step 0: Pre-working setup

In [ ]:
# Install and import the required libraries

In [ ]:
import os
from glob import glob

In [ ]:
import geopandas
import pandas
import fiona
import numpy

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

In [ ]:
import rasterio

In [ ]:
# Identify current working directory and create any shortcuts to the location of the project data to make life simpler

In [ ]:
os.getcwd()

In [ ]:
base_path = 'Z:\\jamaica'

In [ ]:
os.chdir(base_path)

## Step 1: Baseline assessment of existing natural assets

In [ ]:
# Start by mapping out the existing terrestrial landcover and coastal ecosystems - note if there is an error it is likely because the files have been pushed into icloud

# Terrestrial land cover can be obtained from the Forestry Department landcover map supplied by Edson

# Coastal ecosystems encompass mangroves (obtained from World Bank Forces of Nature project), coral reefs and seagrass (both obtained from Edson)

### Terrestrial landcover map

In [ ]:
bauxite = geopandas.read_file(os.path.join(base_path, 'nsmdb-bauxite_reserves.gpkg'))

In [ ]:
bauxite2 = geopandas.read_file("Z:\\jamaica/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="bauxite")

In [ ]:
bauxite

In [ ]:
bauxite.plot()

In [ ]:
landcover = geopandas.read_file(os.path.join(base_path, 'jamaica_land_use_combined.gpkg'))

In [ ]:
landcover

In [ ]:
landcover.groupby('Classify').sum()

In [ ]:
landcover[['area_hectare', 'Classify']].groupby('Classify').sum()

In [ ]:
for classification in landcover.Classify.unique():
    print(classification)

In [ ]:
landcover_classification = pandas.read_csv(os.path.join(base_path, 'config', 'landcover_classification.csv'))
landcover_classification

In [ ]:
landcover_classified = landcover.merge(landcover_classification, on='Classify')

In [ ]:
primary_forest_copy = geopandas.GeoDataFrame(
    {
        "geometry": primary_forest_copy.geometry.unary_union 
    },
    crs=jamaica_crs
)

In [ ]:
f"Format strings can include variables between curly brackets like {len(landcover)}"

In [ ]:
assert len(landcover) == len(landcover_classified), f"Lengths were {len(landcover)} {len(landcover_classified)}"

#### Terrestrial landcover plots

##### All forests

In [ ]:
landcover_classified.all_forest_dry_and_mixed

In [ ]:
landcover_classified.all_forest_dry_and_mixed == 1

In [ ]:
all_forest = landcover_classified[landcover_classified.all_forest_dry_and_mixed == 1]
all_forest.plot

##### Primary Forest

In [ ]:
landcover_classified.primary_forest

In [ ]:
primary_forest = landcover_classified[landcover_classified.primary_forest == 1]
primary_forest.plot

##### Agriculture (all)

In [ ]:
Agriculture_including_plantations = landcover_classified[landcover_classified.Agriculture_including_plantations == 1]
Agriculture_including_plantations.plot

##### Agriculture (excluding plantations)

In [ ]:
Agriculture_excluding_plantations = landcover_classified[landcover_classified.Agriculture_excluding_plantations == 1]
Agriculture_excluding_plantations.plot

##### Agriculture (bare fields)

In [ ]:
Agriculture_bare_fields = landcover_classified[landcover_classified.Agriculture_bare_fields == 1]
Agriculture_bare_fields.plot

##### Dry forests

In [ ]:
dry_forest_all = landcover_classified[landcover_classified.dry_forest_all == 1]
dry_forest_all.plot

##### Wetland

In [ ]:
Wetland = landcover_classified[landcover_classified.Wetland == 1]
Wetland.plot

#### Coastal NbS plots

##### Mangroves

In [ ]:
mangroves = geopandas.read_file(os.path.join(base_path, 'mangroves/commondata/nsdmd/mangroves.shp'))

In [ ]:
mangroves.plot()

##### Coral Reefs

In [ ]:
corals = geopandas.read_file(os.path.join(base_path, 'nsmdb-reefs_jan06_NEPA.gpkg'))

In [ ]:
corals.plot()

In [ ]:
corals.cx[600_000:850_000,600_000:750_000].plot()

##### Seagrass

In [ ]:
seagrass = geopandas.read_file(os.path.join(base_path, 'nsmdb-seagrass.gpkg'))

In [ ]:
seagrass.plot()

In [ ]:
seagrass.cx[600_000:850_000,600_000:750_000].plot()

### Assessing Ecosystem Health

In [ ]:
# Ecosystem health and functioning can be assessed through various proxies (see paper by Key et al (2021))

# Here we will assess via: (1) Connectivity analysis; (2) Fragmentation analysis; (3) Biodiversity Intactness Index; (4) NDVI

#### Terrestrial ecosystem health

##### Forest fragmentation

In [ ]:
# Can we calculate this?

##### Forest connectivity

In [ ]:
# Can we calculate this?

#### Coastal ecosystem health

In [ ]:
# Coastal ecosystem capacity for service delivery (including resilience) is said to be highest when mangroves, seagrass and coral reefs are in proximity. 

# Proximity for maximum functioning depends on species and dispersal range (e.g. fish) - papers report proximities should be within 1000m, 500m and 250m. 

# All of the above proximities will be considered here.

##### Connectivity

In [ ]:
for distance in [250, 500, 1000]:
    mangroves[f"geometry_{distance}m_buffer"] = mangroves.buffer(distance=distance)
    corals[f"geometry_{distance}m_buffer"] = corals.buffer(distance=distance)
    seagrass[f"geometry_{distance}m_buffer"] = seagrass.buffer(distance=distance)

In [ ]:
mangroves

###### 1000m

In [ ]:
mangroves_within_1000m = mangroves \
    .overlay(corals.set_geometry('geometry_1000m_buffer'), how='intersection') \
    .overlay(seagrass.set_geometry('geometry_1000m_buffer'), how='intersection')

In [ ]:
mangroves_within_1000m.plot()

###### 500m

In [ ]:
mangroves_within_500m = mangroves \
    .overlay(corals.set_geometry('geometry_500m_buffer'), how='intersection') \
    .overlay(seagrass.set_geometry('geometry_500m_buffer'), how='intersection')

In [ ]:
mangroves_within_500m.plot()

###### 250m

In [ ]:
mangroves_within_250m = mangroves \
    .overlay(corals.set_geometry('geometry_250m_buffer'), how='intersection') \
    .overlay(seagrass.set_geometry('geometry_250m_buffer'), how='intersection')

In [ ]:
mangroves_within_250m.plot()

##### Biodiversity Intactness Index

In [ ]:
Biodiversity_intactness_index = rasterio.open('/Users/robynhaggis/Documents/FCDO - Jamaica/Input data/lbii-jamaica.tif')

In [ ]:
Biodiversity_intactness_index

In [ ]:
plt.imshow(Biodiversity_intactness_index.read(1), cmap='viridis')
plt.show()

### Protected Areas including Forest Reserves

#### Protected areas (excluding forest reserves)

In [ ]:
protected_areas = geopandas.read_file(os.path.join(base_path, 'nsmdb-protected_areas.gpkg'))

In [ ]:
protected_areas.plot()

In [ ]:
protected_areas = geopandas.read_file("/Users/robynhaggis/Documents/FCDO - Jamaica/Input data/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="protected_areas")

In [ ]:
protected_areas.columns

In [ ]:
protected_areas

In [ ]:
protected_areas.plot(column="LAYER", cmap="tab10", legend=True)

#### Forest reserves

In [ ]:
forest_reserves = geopandas.read_file("Z:\\jamaica/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="forest_reserves")

In [ ]:
forest_reserves

In [ ]:
forest_reserves.columns

In [ ]:
forest_reserves.plot()

##### Forest reserves and protected areas 

In [ ]:
protected_areas_and_forest_reserves = forest_reserves.overlay(protected_areas, how='union')

In [ ]:
protected_areas_and_forest_reserves.plot()

In [ ]:
fiona.listlayers("Z:\\jamaica/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb")

### Assessing by water catchment area (hydrobasins)

#### Locating hydrobasins

In [ ]:
hydrobasins = geopandas.read_file(os.path.join(base_path, "Input data","hydrobasins.gpkg"), layer='hybas_lake_na_lev08_v1c')

In [ ]:
fiona.listlayers("/Users/robynhaggis/Documents/FCDO - Jamaica/Input data/hydrobasins.gpkg")

In [ ]:
hydrobasins

In [ ]:
hydrobasins.MAIN_BAS = hydrobasins.MAIN_BAS.apply(str)
hydrobasins.plot(column='MAIN_BAS', cmap='tab20', legend='true')

#### Calculating extent (area) of different natural landcover in hydrobasins 

In [ ]:
hydrobasins['area_km2'] = hydrobasins.geometry.area / 1e6

##### Extent of forest in each catchment

In [ ]:
hydrobasins_with_all_forest = hydrobasins.overlay(all_forest, how='intersection')

In [ ]:
hydrobasins_with_all_forest['all_forest_area_km2'] = hydrobasins_with_all_forest.geometry.area / 1e6

In [ ]:
hydrobasins_with_all_forest_area = hydrobasins_with_all_forest.groupby('HYBAS_ID').sum()

In [ ]:
hydrobasins_with_all_forest_area[['all_forest_area_km2']]

#### Percentage of catchment containing forest

In [ ]:
hydrobasins = hydrobasins.set_index('HYBAS_ID').join(hydrobasins_with_all_forest_area[['all_forest_area_km2']])

In [ ]:
hydrobasins['all_forest_area_perc'] = hydrobasins.all_forest_area_km2 / hydrobasins.area_km2

In [ ]:
hydrobasins['percent_of_total_forest_area_in_basin'] = hydrobasins.all_forest_area_km2 / hydrobasins.all_forest_area_km2.sum()

In [ ]:
hydrobasins.percent_of_total_forest_area_in_basin.sum()

In [ ]:
hydrobasins

##### Extent of Primary forest in each catchment

In [ ]:
hydrobasins_with_primary_forest = hydrobasins.overlay(primary_forest, how='intersection')

In [ ]:
hydrobasins_with_primary_forest['primary_forest_area_km2'] = hydrobasins_with_primary_forest.geometry.area / 1e6

In [ ]:
hydrobasins_with_primary_forest_area = hydrobasins_with_primary_forest.groupby('HYBAS_ID')['primary_forest_area_km2'].sum()

In [ ]:
hydrobasins_with_primary_forest_area[['primary_forest_area_km2']]

#### Percentage of catchment containing primary forest

In [ ]:
hydrobasins = hydrobasins.set_index('HYBAS_ID').join(hydrobasins_with_primary_forest_area[['primary_forest_area_km2']])

In [ ]:
hydrobasins['primary_forest_area_perc'] = hydrobasins.primary_forest_area_km2 / hydrobasins.area_km2

In [ ]:
hydrobasins['percent_of_primary_forest_area_in_basin'] = hydrobasins.primary_forest_area_km2 / hydrobasins.primary_forest_area_km2.sum()

In [ ]:
hydrobasins.percent_of_primary_forest_area_in_basin.sum()

#### Extent of Agriculture (all) in each catchment

In [ ]:
hydrobasins_with_Agriculture_including_plantations = hydrobasins.overlay(Agriculture_including_plantations, how='intersection')

In [ ]:
hydrobasins_with_Agriculture_including_plantations['Agriculture_including_plantations_km2'] = hydrobasins_with_Agriculture_including_plantations.geometry.area / 1e6

In [ ]:
hydrobasins_with_Agriculture_including_plantations = hydrobasins_with_Agriculture_including_plantations.groupby('HYBAS_ID').sum()

In [ ]:
hydrobasins_with_Agriculture_including_plantations[['Agriculture_including_plantations_km2']]

#### Percentage of catchment containing Agriculture

In [ ]:
hydrobasins = hydrobasins.set_index('HYBAS_ID').join(hydrobasins_with_Agriculture_including_plantations[['Agriculture_including_plantations_km2']])

## Step 2: assessment of infrastructure-related ecosystem services

#### Land cover CN numbers: impact on water flow velocity

In [ ]:
# CN numbers have been mostly based on 'fair' conditions (not poor or good)
# These could be improved through NbS which improve management 

##### 'A' category CN numbers

In [ ]:
CN_Fair_A = landcover_classified[landcover_classified.CN_Fair_A == 1]
CN_Fair_A.plot

##### 'B' category CN numbers

In [ ]:
CN_Fair_B = landcover_classified[landcover_classified.CN_Fair_B == 1]
CN_Fair_B.plot

##### 'C' category CN numbers

In [ ]:
CN_Fair_C = landcover_classified[landcover_classified.CN_Fair_C == 1]
CN_Fair_C.plot

##### 'D' category CN numbers

In [ ]:
CN_Fair_D = landcover_classified[landcover_classified.CN_Fair_D == 1]
CN_Fair_D.plot

#### Surface roughness impact on water flow velocity

In [ ]:
Surface_roughness = landcover_classified[landcover_classified.Surface_roughness == 1]
Surface_roughness.plot

#### Soil permeability impact on above-ground and below-ground water flows

In [ ]:
soil = geopandas.read_file("/Users/robynhaggis/Documents/FCDO - Jamaica/Input data/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="soils")

In [ ]:
soil.columns

In [ ]:
soil.plot(column="TEXTURE1", cmap="tab10", legend=True)

In [ ]:
soils = pandas.DataFrame(soil, columns =["NAME", "CONTROL", "MINEROLOGY", "TEXTURE1", "REGION", "Shape_Length", "Shape_Area", "geometry", "soil class", "area"])

In [ ]:
soil.TEXTURE1.unique()

In [ ]:
soils

In [ ]:
def give_soil_class(TEXTURE1_type):
    if TEXTURE1_type == "fine sandy loam":
        return "A"
    elif TEXTURE1_type == "gravely sandy loam":
        return "A"
    elif TEXTURE1_type == "sandy loam":
        return "A"
    elif TEXTURE1_type == "silt":
        return "A"
    elif TEXTURE1_type == "stony loam":
        return "A"
    elif TEXTURE1_type == "stony sandy loam":
        return "A"
    elif TEXTURE1_type == "gravely loam":
        return "B"
    elif TEXTURE1_type == "loam":
        return "B"
    elif TEXTURE1_type == "chanery sandy clay loam":
        return "C"
    elif TEXTURE1_type == "fine sandy clay loam":
        return "C"
    elif TEXTURE1_type == "gravely sandy clay loam":
        return "C"
    elif TEXTURE1_type == "grravely sandy clay":
        return "C"
    elif TEXTURE1_type == "sandy clay":
        return "C"
    elif TEXTURE1_type == "sandy clay loam":
        return "C"
    elif TEXTURE1_type == "stony fine sandy clay loam":
        return "C"
    elif TEXTURE1_type == "stony sandy clay loam":
        return "C"
    elif TEXTURE1_type == "clay":
        return "D"
    elif TEXTURE1_type == "chanery clay":
        return "D"
    elif TEXTURE1_type == "chanery clay loam":
        return "D"
    elif TEXTURE1_type == "clay loam":
        return "D"
    elif TEXTURE1_type == "gravely clay loam":
        return "D"
    elif TEXTURE1_type == "silty clay":
        return "D"
    elif TEXTURE1_type == "silty clay loam":
        return "D"
    elif TEXTURE1_type == "stony clay":
        return "D"
    elif TEXTURE1_type == "stony clay loam":
        return "D"
    else:
        return "unknown"

In [ ]:
soils["soil class"] = soils.apply(lambda x: give_type(x["TEXTURE1"]), axis=1)
print(soils)

#### Riparian landcover assessment

##### River locations

In [ ]:
headwater_rivers = geopandas.read_file("/Users/robynhaggis/Documents/FCDO - Jamaica/Input data/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="ja_fw_target_headwater_streams_18jan06")

In [ ]:
major_rivers = geopandas.read_file("/Users/robynhaggis/Documents/FCDO - Jamaica/Input data/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="Major_Rivers")

In [ ]:
headwater_rivers.plot()

In [ ]:
major_rivers.plot()

##### Riparian buffer strips

In [ ]:
# 50m buffer either side of rivers

In [ ]:
headwater_riparian = headwater_rivers.buffer(distance = 50)

In [ ]:
headwater_riparian.plot()

In [ ]:
major_river_riparian = major_rivers.buffer(distance = 50)

In [ ]:
major_river_riparian.plot()

## Step 3: assessment of threats to ecosystems and their services

### Mining risk: assessment of bauxite reserves and their threat to land cover

In [ ]:
# First map the location of bauxite reserves. 
# Edson has provided two options: Bauxite Bearing Areas and Bauxite Reserves. They appear similar size and similar location, although there are differences. 
# For now Bauxite Bearing Areas has been selected.

In [ ]:
Mining = landcover_classified[landcover_classified.Mining == 1]
Mining.plot

In [ ]:
bauxite = geopandas.read_file("/Users/robynhaggis/Documents/FCDO - Jamaica/Input data/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb", layer="Bauxite_Bearing_Areas")

In [ ]:
bauxite

In [ ]:
bauxite.plot()

In [ ]:
bauxite.crs

In [ ]:
# Intersect Bauxite Bearing Areas with terrestrial landcover to identify which natural ecosystems are located on bauxite bearing areas, and which may be at risk if they are mined

In [ ]:
bauxite_landcover = bauxite.overlay(landcover, how='intersection')

In [ ]:
bauxite_landcover.plot(column="Classify", cmap="tab10", legend='True')

### Risk of urban encroachment

In [ ]:
# 100m buffer around NbS
# Calculate percentage of the 100m buffer which is urban
# Calculate the total boundary length of NbS which is bordered by urban
# Calculate the percentage boundary length of NbS which is bordered by urban

In [ ]:
Primary_forest_buffered = primary_forest.buffer(distance = 50)
Primary_forest_buffered.plot()

In [ ]:
Primary_forest_buffered.crs

In [ ]:
landcover.crs

In [ ]:
Primary_forest_landcover = Primary_forest_buffered.overlay(landcover, how='intersection')

### Risk of agricultural encroachment

In [ ]:
# 100m buffer around NbS
# Calculate percentage of the 100m buffer which is agriculture
# Calculate the total boundary length of NbS which is bordered by agriculture
# Calculate the percentage boundary length of NbS which is bordered by agriculture

### Risk of invasive species: bamboo

In [ ]:
# Plot just bamboo on a map

In [ ]:
Bamboo_including_mixed = landcover_classified[landcover_classified.Bamboo_including_mixed == 1]
Bamboo_including_mixed.plot

## Step 5: assessment of costs and benefits

### Benefits

#### SDGs

##### Terrestrial

###### Number of SDG Goals influenced

In [ ]:
# Assign a SDGs column to the landcover and coastal ecosystems
# landcover_SDG_Goals = landcover_SDG_Goals.lower() for making things lowercase

In [ ]:
SDG_Goals= landcover_classified[landcover_classified.SDG_Goals == 1]
SDG_Goals.plot

###### Number of SDG targets influenced

In [ ]:
SDG_Targets= landcover_classified[landcover_classified.SDG_Targets == 1]
SDG_Targets.plot

###### Percentage of SDG targets influenced

In [ ]:
# Calculating the percentage of SDG targets influenced by land cover type

In [ ]:
Percentage_of_SDG_targets= landcover_classified[landcover_classified.Percentage_of_SDG_targets == 1]
Percentage_of_SDG_targets.plot

#### Paris Agreement

In [ ]:
# Benefits in terms of progress on the Paris Agreement will be analysed via mitigation and adaptation outcomes.

##### Mitigation

In [ ]:
# Mitigation co-benefits will be analysed via assessment of above-ground and below-ground carbon.
# This is currently analysed via total 'potential' carbon stocks, not annual carbon sequestration rates. 
# Carbon stocks will be analysed only from the perspective of natural assets.
# Below-ground carbon stocks for built infrastructure will not be assessed.

###### Above-ground carbon

In [ ]:
# All values are in tonnes per hectare (t/ha)

In [ ]:
Aboveground_Carbon= landcover_classified[landcover_classified.Aboveground_Carbon == 1]
Aboveground_Carbon.plot

###### Below-ground carbon

In [ ]:
# All values are in tonnes per hectare (t/ha)

In [ ]:
Belowground_Carbon= landcover_classified[landcover_classified.Belowground_Carbon == 1]
Belowground_Carbon.plot

#### Livelihoods

In [ ]:
# Can we work this out by asking it to divide the number of jobs by the number of hectares?
# We would need to add on an extent column or create a new dataframe

#### Time to implement

In [ ]:
# This will need to be a separate dataframe probably as it depends on the type of NbS (e.g. protect v restore)

## Step 6: prioritization framework

In [ ]:
# Add new dataframe?

# Appendix

In [ ]:
fiona.listlayers("/Users/robynhaggis/Documents/FCDO - Jamaica/Input data/GWP_Jamaica_NSP_Master_Geodatabase_v01.gdb")